# ENGEE319 Lab 4 Solar Radiation Prediction

In [38]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import optuna
import xgboost as xgb

from sklearn.metrics import r2_score
import re

In [ ]:
rad_df = pd.read_csv('solar_radiation.csv')
# sun_df = pd.read_csv('sunshine.csv')

In [ ]:
rad_df.info()
# sun_df.info()

In [ ]:
print(f"Radiation total missing values: {rad_df.isna().sum().sum()}")
# print(f"Sunshine total missing values: {sun_df.isna().sum().sum()}")

## Feature Engineering

In [43]:
df = df.drop(
    [
        'Frequency [D/H]',
        'Data Type',
        'Data Source',
        'PERIOD [hrs]'
    ],
    axis=1
)

In [44]:
df['timeNZST'] = pd.to_datetime(df['Observation time UTC']).dt.tz_convert('Pacific/Auckland')
df = df.drop('Observation time UTC', axis=1)

In [45]:
df['year'] = df['timeNZST'].dt.year
df['month'] = df['timeNZST'].dt.month
df['day'] = df['timeNZST'].dt.day
df['hour'] = df['timeNZST'].dt.hour

df = df.drop('timeNZST', axis=1)

In [ ]:
df.info()

## Splitting/Scaling

In [47]:
y = df['Radiation [MJ/m2]'].copy()
X = df.drop('Radiation [MJ/m2]', axis=1).copy()

In [48]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=67
    )
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    train_size=0.8,
    random_state=69
    )

In [ ]:
print(X_train.shape[0])
print(X_val.shape[0])
print(X_test.shape[0])

In [51]:
# Convert to DMatrix form for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

## Training

In [52]:
def get_model_rmse(params):
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        evals=[(dval, 'eval')],
        early_stopping_rounds=10,
        verbose_eval=0
    )

    results = model.eval(dval)

    rmse = re.search(r'[\d.]+$', results)
    rmse = float(rmse.group())
    
    return rmse

In [ ]:
def objective(trial):
    learning_rate = trial.suggest_loguniform('learning_rate', 0.00001, 1.0)
    max_depth = trial.suggest_int('max_depth', 4, 8)
    l1_reg = trial.suggest_loguniform('l1_reg', 0.00001, 10.0)
    l2_reg = trial.suggest_loguniform('l2_reg', 0.00001, 10.0)

    params = {
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'alpha': l1_reg,
        'lambda': l2_reg
    }

    return get_model_rmse(params)

In [ ]:
study = optuna.create_study()
study.optimize(objective, n_trials=100, show_progress_bar=True)

In [ ]:
best = study.best_params
best

In [ ]:
model = xgb.train(
    best,
    dtrain,
    num_boost_round=3000,
    evals=[(dval, 'eval')],
    early_stopping_rounds=10
    )

## Results

In [61]:
y_true = np.array(y_test, dtype=float)
y_pred = np.array(model.predict(dtest), dtype=float)

In [ ]:
r2 = r2_score(y_true, y_pred)
print(f"R^2 Score: {r2:.2f}")